# Start here: how to use this mini-course

**FQCP 2026 · Bayesian parameter estimation for compact binaries**

> Self-contained and designed for Google Colab. Run top to bottom; **Extension** cells may be skipped live.


## Goal

By the end you should be able to:

1. explain a posterior as **prior × likelihood**, up to normalisation;
2. construct the PSD-weighted Gaussian/Whittle likelihood used in GW inference;
3. recognise how CBC parameters change a waveform and infer a chirp mass;
4. explain why population selection and LISA source overlap require larger models.

### Live route

Open notebooks `01`, `02`, and `04`. Notebook `03` is a short live section if time permits and a useful follow-up otherwise.

\[
\theta \longrightarrow h(f;\theta) \longrightarrow d(f)-h(f;\theta)
\longrightarrow p(d\mid\theta) \longrightarrow p(\theta\mid d).
\]

The same map appears in every chapter. Only the signal, noise model, dimension, and computational strategy change.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
rng = np.random.default_rng(20260817)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["animation.html"] = "jshtml"
print("Running in Colab:", IN_COLAB)

Running in Colab: False


## Reusable helper file

Colab runtimes are temporary, but they can fetch a plain Python module from GitHub. The next cell downloads `fqcp_helpers.py` in Colab and imports the local copy during development. For a released course, replace `main` in the URL with a version tag such as `v1.0.0` so old notebooks remain reproducible.

In [2]:
# One-file helper pattern: download from GitHub in Colab, import locally otherwise.
import sys
import urllib.request
from pathlib import Path

HELPER_URL = "https://raw.githubusercontent.com/nz-gravity/FQCP2026_GW_data_analysis/main/fqcp_helpers.py"
if IN_COLAB:
    urllib.request.urlretrieve(HELPER_URL, "fqcp_helpers.py")
else:
    candidates = [Path.cwd(), Path.cwd()/"FQCP2026_GW_data_analysis", Path.cwd().parent]
    helper_parent = next((path for path in candidates if (path/"fqcp_helpers.py").exists()), None)
    if helper_parent is None:
        raise FileNotFoundError("Could not locate fqcp_helpers.py")
    sys.path.insert(0, str(helper_parent))

from fqcp_helpers import equal_tailed_interval, frequency_inner_product, normalise_log_density
print("Course helpers loaded from", HELPER_URL if IN_COLAB else helper_parent)

Course helpers loaded from /Users/avi/Documents/projects/WORKSHOP/FQCP2026_GW_data_analysis


In [3]:
import scipy, matplotlib
print("NumPy", np.__version__)
print("SciPy", scipy.__version__)
print("Matplotlib", matplotlib.__version__)
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)
print("Environment check passed.")

NumPy 2.4.4
SciPy 1.15.2
Matplotlib 3.10.9
Environment check passed.


## Vocabulary

| Term | Meaning |
| --- | --- |
| data, `d` | detector output: signal plus noise |
| parameters, `theta` | quantities we want to learn |
| prior | plausible values before this dataset |
| likelihood | compatibility of parameters with data and noise model |
| posterior | updated distribution after seeing the data |
| PSD | noise power as a function of frequency |
| evidence | probability of the data under a whole model |

<details><summary>Why not report only a best fit?</summary>

A best fit does not show uncertainty, degeneracies, multiple solutions, or prior sensitivity. A posterior can show all four.

</details>

## Before teaching

- Run all notebooks in a fresh Colab runtime.
- Open each notebook in a browser tab before the session.
- Animations use JavaScript HTML and do not need `ffmpeg`.
- Re-test the pinned ripple installation shortly before the workshop.